# Практика: логирование CatBoost в MLflow

Этот ноутбук показывает полный цикл:
1. Подключение к MLflow Tracking Server и MinIO
2. Подготовка данных
3. Обучение модели CatBoostClassifier
4. Логирование параметров, метрик и модели в MLflow
5. Загрузка модели из реестра и инференс

## 0) Установка зависимостей (если нужно)
Если запускаете впервые, раскомментируйте строку ниже и выполните ячейку.

In [ ]:
# !uv add catboost mlflow scikit-learn pandas python-dotenv

## 1) Импорты и настройки окружения

In [ ]:
import os
import warnings

import pandas as pd
from dotenv import load_dotenv
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

import mlflow
import mlflow.catboost
from mlflow.models import infer_signature

warnings.filterwarnings("ignore")
load_dotenv()

In [ ]:
# Для локального стенда из docker-compose
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")

# `minio` резолвится только внутри docker-сети. Для локального ноутбука нужен localhost.
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

# Если на сервере включена basic-auth, задайте логин/пароль
# os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "admin")
# os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "password")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("MLflow URI:", mlflow.get_tracking_uri())

In [ ]:
# Быстрая проверка подключения
mlflow.search_experiments(max_results=3)

## 2) Подготовка датасета

In [ ]:
df = pd.read_csv("data/apple_quality.csv")
df = df.dropna().copy()
df["target"] = (df["Quality"] == "good").astype(int)
df = df.drop(columns=["Quality", "A_id"])

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("train:", X_train.shape, "test:", X_test.shape)

## 3) Настройка эксперимента и запуск run

In [ ]:
from mlflow.tracking import MlflowClient

experiment_name = "students-catboost-demo-proxy"
artifact_location = "mlflow-artifacts:/"
client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
    print("Created experiment:", exp_id)
else:
    exp_id = exp.experiment_id
    print("Using existing experiment:", exp_id, "artifact_location=", exp.artifact_location)

mlflow.set_experiment(experiment_name)

registered_model_name = "students_catboost_apple_quality"

params = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.05,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "verbose": False,
    "random_seed": 42
}


In [ ]:
with mlflow.start_run(experiment_id=exp_id):
    print("Artifact URI for this run:", mlflow.get_artifact_uri())
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred)),
        "roc_auc": float(roc_auc_score(y_test, proba)),
        "log_loss": float(log_loss(y_test, proba)),
    }

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_test, proba)

    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_test.head(5),
        registered_model_name=registered_model_name,
    )

    mlflow.log_input(
        mlflow.data.from_pandas(df, source="data/apple_quality.csv"),
        context="training",
    )

    # Маркируем текущую версию как PRD в реестре моделей
    client = MlflowClient()
    new_version = model_info.registered_model_version
    client.set_model_version_tag(registered_model_name, new_version, "env", "PRD")
    client.set_registered_model_alias(registered_model_name, "prd", new_version)

    run_id = mlflow.active_run().info.run_id
    print("Run ID:", run_id)
    print("Registered model version:", new_version)
    print("Alias 'prd' points to version:", new_version)
    print("Metrics:", metrics)


Если видите ошибку `experiment 0 ... deleted`:

Это означает, что MLflow пытается писать в удаленный дефолтный эксперимент.
В этом ноутбуке run запускается с явным `experiment_id=exp_id`, поэтому просто перезапустите kernel и выполните ячейки сверху вниз.

## 4) Проверка результатов в MLflow

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=[experiment_name],
    order_by=["metrics.roc_auc DESC"],
)
runs_df[["run_id", "metrics.accuracy", "metrics.f1", "metrics.roc_auc", "artifact_uri"]].head()

In [ ]:
# Показать запуски
mlflow.search_runs(experiment_names=[experiment_name])

## 5) Загрузка модели из Model Registry по тегу PRD (alias `prd`)

In [ ]:
loaded_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@prd")

sample_pred = loaded_model.predict(X_test.head(3))
sample_pred
